In [25]:
import sys
import os
from pathlib import Path
from src.utils import normalize_columns

root_path = str(Path(os.getcwd()).parent)
if root_path not in sys.path:
    sys.path.append(root_path)


In [26]:
import pandas as pd
from src.utils import normalize_columns

cursos = ["Técnico em Eletrotécnica", "Mecânico de Manutenção Industrial", "Web Design Avançado"]
df_cursos = pd.DataFrame(cursos, columns=["nome_curso"])

df_cursos = normalize_columns(df_cursos, ["nome_curso"], 'original')
print(df_cursos)

Iniciando normalização local de 1 colunas...
Normalizando coluna: nome_curso...
Progresso: 3/3 linhas processadas.
Processo concluído com sucesso!
                          nome_curso             nome_curso_original
0           Técnico em Eletrotécnica           tecnico eletrotecnica
1  Mecânico de Manutenção Industrial  mecanico Manutencao industrial
2                Web Design Avançado              web design avancar


In [27]:
cursos_mock = [
    "Técnico em Eletrotécnica",
    "Curso de Aperfeiçoamento em Mecânica de Manutenção Industrial",
    "Formação em Web Design Avançado",
    "Capacitação para Cozinheiro Industrial"
]

cbo_mock = {
    "721215": "Mecânico de manutenção de máquinas em geral",
    "313105": "Eletrotécnico",
    "317110": "Desenvolvedor de sistemas (Web Design)",
    "513205": "Cozinheiro geral"
}

df_cursos = pd.DataFrame(cursos_mock, columns=["nome_curso"])
df_cbo = pd.DataFrame(list(cbo_mock.items()), columns=["codigo", "ocupacao"])

In [13]:
df_cursos = normalize_columns(df_cursos, ["nome_curso"], 'limpo')
df_cbo = normalize_columns(df_cbo, ["ocupacao"], 'limpo')

Iniciando normalização local de 1 colunas...
Normalizando coluna: nome_curso...
Progresso: 4/4 linhas processadas.
Processo concluído com sucesso!
Iniciando normalização local de 1 colunas...
Normalizando coluna: ocupacao...
Progresso: 4/4 linhas processadas.
Processo concluído com sucesso!


In [34]:
import os
from optimum.onnxruntime import ORTModelForFeatureExtraction
from transformers import AutoTokenizer

model_id = "neuralmind/bert-base-portuguese-cased"
output_path = "../resources/models/bertimbau_onnx"

print("Baixando e convertendo o BERTimbau (isso pode demorar)...")
model = ORTModelForFeatureExtraction.from_pretrained(model_id, export=True)
tokenizer = AutoTokenizer.from_pretrained(model_id)

model.save_pretrained(output_path)
tokenizer.save_pretrained(output_path)

file_size = os.path.getsize(f"{output_path}/model.onnx") / (1024 * 1024)
if file_size > 100:
    print(f"✅ Sucesso! O arquivo model.onnx tem {file_size:.2f} MB.")
else:
    print(f"❌ Erro: O arquivo ainda está muito pequeno ({file_size:.2f} MB).")

Baixando e convertendo o BERTimbau (isso pode demorar)...


`torch_dtype` is deprecated! Use `dtype` instead!
/home/gabrielsousa/miniconda3/envs/cbo-backend/lib/python3.10/site-packages/transformers/modeling_attn_mask_utils.py:196: TracerWarning: torch.tensor results are registered as constants in the trace. You can safely ignore this warning if you use this function to create tensors out of constant variables that would be the same every time you call this function. In any other case, this might cause the trace to be incorrect.
  inverted_mask = torch.tensor(1.0, dtype=dtype) - expanded_mask


✅ Sucesso! O arquivo model.onnx tem 413.44 MB.


In [2]:
import onnxruntime as ort

model_path = "../resources/models/bertimbau_onnx"
sess_options = ort.SessionOptions()
sess_options.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL

session = ort.InferenceSession(
    f"{model_path}/model.onnx",
    sess_options,
    providers=['CPUExecutionProvider']
)

In [36]:
from optimum.onnxruntime import ORTModelForFeatureExtraction
from transformers import AutoTokenizer
import os

model_id = "neuralmind/bert-base-portuguese-cased"
output_path = "../resources/models/bertimbau_onnx"

os.makedirs(output_path, exist_ok=True)

print(f"Iniciando a exportação do {model_id} para ONNX...")

model = ORTModelForFeatureExtraction.from_pretrained(model_id, export=True)
tokenizer = AutoTokenizer.from_pretrained(model_id)

model.save_pretrained(output_path)
tokenizer.save_pretrained(output_path)

print(f"Exportação concluída com sucesso em: {output_path}")

Iniciando a exportação do neuralmind/bert-base-portuguese-cased para ONNX...


/home/gabrielsousa/miniconda3/envs/cbo-backend/lib/python3.10/site-packages/transformers/modeling_attn_mask_utils.py:196: TracerWarning: torch.tensor results are registered as constants in the trace. You can safely ignore this warning if you use this function to create tensors out of constant variables that would be the same every time you call this function. In any other case, this might cause the trace to be incorrect.
  inverted_mask = torch.tensor(1.0, dtype=dtype) - expanded_mask


Exportação concluída com sucesso em: ../app/resources/models/bertimbau_onnx


In [41]:
import onnxruntime as ort
from transformers import AutoTokenizer
import numpy as np

model_path = "../resources/models/bertimbau_onnx"
tokenizer = AutoTokenizer.from_pretrained(model_path)

sess_options = ort.SessionOptions()
sess_options.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
sess_options.intra_op_num_threads = 4

session = ort.InferenceSession(
    f"{model_path}/model.onnx",
    sess_options,
    providers=['CPUExecutionProvider']
)

model_inputs = [i.name for i in session.get_inputs()]
print(f"--> Inputs esperados pelo ONNX: {model_inputs}")

col_cbo = 'ocupacao_limpo' if 'ocupacao_limpo' in df_cbo.columns else 'ocupacao'
col_cursos = 'nome_curso_limpo' if 'nome_curso_limpo' in df_cursos.columns else 'nome_curso'

def get_embeddings_in_batch(dataframe, column_name, batch_size=16):
    texts = dataframe[column_name].astype(str).tolist()
    all_embeddings = []

    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]

        inputs = tokenizer(
            batch,
            return_tensors="np",
            padding=True,
            truncation=True,
            max_length=512
        )

        input_feed = {name: inputs[name] for name in model_inputs if name in inputs}

        outputs = session.run(None, input_feed)

        token_embeddings = outputs[0]
        mask = inputs["attention_mask"][:, :, np.newaxis]

        sum_embeddings = np.sum(token_embeddings * mask, axis=1)
        sum_mask = np.maximum(np.sum(mask, axis=1), 1e-9)
        batch_mean = sum_embeddings / sum_mask

        all_embeddings.append(batch_mean)

    return np.vstack(all_embeddings).astype('float32')

print(f"--> Iniciando extração para CBO...")
vetores_cbo = get_embeddings_in_batch(df_cbo, col_cbo)

print(f"--> Iniciando extração para Cursos...")
vetores_cursos = get_embeddings_in_batch(df_cursos, col_cursos)

print(f"✅ Sucesso! Vetores CBO: {vetores_cbo.shape}, Vetores Cursos: {vetores_cursos.shape}")

--> Inputs esperados pelo ONNX: ['input_ids', 'attention_mask', 'token_type_ids']
--> Iniciando extração para CBO...
--> Iniciando extração para Cursos...
✅ Sucesso! Vetores CBO: (4, 768), Vetores Cursos: (4, 768)


In [43]:
import faiss

faiss.normalize_L2(vetores_cbo)
faiss.normalize_L2(vetores_cursos)

index = faiss.IndexFlatIP(768)

index.add(vetores_cbo)

distancias, indices = index.search(vetores_cursos, k=1)

print("-" * 50)
print(f"{'CURSO ANALISADO':<40} | {'CBO PROVÁVEL':<40} | {'SIMILARIDADE'}")
print("-" * 100)

for i, curso in enumerate(df_cursos[col_cursos]):
    idx_cbo = indices[i][0]
    score = distancias[i][0]
    cbo_encontrada = df_cbo.iloc[idx_cbo]['ocupacao']
    codigo_cbo = df_cbo.iloc[idx_cbo]['codigo']

    print(f"{curso[:38]:<40} | {cbo_encontrada[:38]:<40} | {score:.4f}")

print("-" * 100)

faiss.write_index(index, "../app/resources/models/cbo_index.faiss")

# Para carregar depois:
# index = faiss.read_index("../app/resources/models/cbo_index.faiss")

--------------------------------------------------
CURSO ANALISADO                          | CBO PROVÁVEL                             | SIMILARIDADE
----------------------------------------------------------------------------------------------------
Técnico em Eletrotécnica                 | Eletrotécnico                            | 0.8734
Curso de Aperfeiçoamento em Mecânica d   | Mecânico de manutenção de máquinas em    | 0.7106
Formação em Web Design Avançado          | Desenvolvedor de sistemas (Web Design)   | 0.8145
Capacitação para Cozinheiro Industrial   | Cozinheiro geral                         | 0.6973
----------------------------------------------------------------------------------------------------
